In [152]:
# On réutilise les fonctions du script de collecte
import sys
sys.path.append("collecte")

from collecte import collecter_historique

# Chargement de l'historique en mémoire
df = collecter_historique()
df.head()

,time,wave_height,wave_period,wave_direction,swell_wave_height,swell_wave_period,sea_surface_temperature,temperature_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation,relative_humidity_2m,dew_point_2m,cloud_cover_low
0,2024-01-01T00:00,0.96,10.30,314,0.86,8.00,16.9,17.1,8.2,113,16.2,0.0,25,-3.4,0
1,2024-01-01T01:00,0.98,10.65,314,0.82,8.10,16.9,16.6,5.5,122,17.6,0.0,31,-0.5,0
2,2024-01-01T02:00,1.04,11.20,315,0.90,8.80,16.9,15.6,5.7,108,13.7,0.0,37,1.0,0
3,2024-01-01T03:00,1.12,11.90,317,0.98,9.85,16.9,14.7,5.1,82,11.5,0.0,34,-1.1,0
4,2024-01-01T04:00,1.18,12.50,318,1.06,10.75,16.9,14.5,4.1,52,8.6,0.0,30,-2.8,0


In [153]:
# dimension, type et valeurs manquantes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   time                     17544 non-null  str    
 1   wave_height              17544 non-null  float64
 2   wave_period              17544 non-null  float64
 3   wave_direction           17544 non-null  int64  
 4   swell_wave_height        17544 non-null  float64
 5   swell_wave_period        17544 non-null  float64
 6   sea_surface_temperature  17232 non-null  float64
 7   temperature_2m           17544 non-null  float64
 8   wind_speed_10m           17544 non-null  float64
 9   wind_direction_10m       17544 non-null  int64  
 10  wind_gusts_10m           17544 non-null  float64
 11  precipitation            17544 non-null  float64
 12  relative_humidity_2m     17544 non-null  int64  
 13  dew_point_2m             17544 non-null  float64
 14  cloud_cover_low          17544 no

17 544 lignes, 11 colonnes, aucune valeur manquante. Deux ans de données horaires complètes, pas de nettoyage nécessaire.

In [154]:
import pandas as pd

# On extrait le mois pour regrouper
df["mois"] = pd.to_datetime(df["time"]).dt.month

# Conditions moyennes par mois
df.groupby("mois")[["wave_height", "swell_wave_period", "wind_speed_10m"]].mean().round(2)

,wave_height,swell_wave_period,wind_speed_10m
mois,,,
1,1.38,10.04,8.76
2,1.44,10.12,10.06
3,1.62,8.70,11.18
4,1.25,7.66,12.13
5,1.07,6.41,10.36
6,0.98,5.92,10.60
7,0.99,5.67,8.29
8,1.00,6.15,7.69
9,0.95,6.52,7.47


La houle suit un cycle net : vagues de 1,3 à 1,6 m et période de 9 à 10 s de décembre à mars, contre 1 m et 5,7 s en été. Une période longue signale une houle venue du large, donc organisée. L’hiver est la saison du surf.

In [155]:
# Direction moyenne du vent par mois (en degrés)
df.groupby("mois")["wind_direction_10m"].mean().round(0)

mois
1     153.0
2     171.0
3     196.0
4     221.0
5     245.0
6     259.0
7     247.0
8     229.0
9     221.0
10    196.0
11    154.0
12    159.0
Name: wind_direction_10m, dtype: float64

Attention : une moyenne de directions n’a pas de sens, les degrés sont circulaires (la moyenne de 350° et 10° donne 180°, soit l’inverse de la réalité). On remplace par un comptage d’heures favorables.

In [156]:
# Vent offshore à Taghazout : venant du secteur est (45° à 135°)
df["offshore"] = df["wind_direction_10m"].between(45, 135)

# Part d'heures offshore par mois, en %
(df.groupby("mois")["offshore"].mean() * 100).round(1)

mois
1     36.5
2     27.7
3     18.1
4     11.7
5      5.4
6      5.3
7      6.1
8      9.4
9     12.6
10    14.8
11    35.6
12    32.4
Name: offshore, dtype: float64

Vent offshore (venant des terres, favorable) 30 à 36 % du temps de novembre à janvier, contre 5 % en mai-juin. Les deux critères convergent vers l’hiver.

In [157]:
# On extrait l'heure
df["heure"] = pd.to_datetime(df["time"]).dt.hour

# Part d'heures offshore et vent moyen, par heure de la journée
df.groupby("heure").agg(
    offshore_pct=("offshore", lambda x: round(x.mean() * 100, 1)),
    vent_moyen=("wind_speed_10m", "mean"),
).round(1)


,offshore_pct,vent_moyen
heure,,
0,30.2,6.8
1,31.6,6.1
2,31.7,5.9
3,36.3,5.6
4,36.5,5.2
5,35.8,5.1
6,35.6,5.2
7,38.9,5.2
8,39.3,5.4


Effet de brise thermique très marqué : 39 % d’heures offshore à 8 h, 0,1 % à 17 h, et un vent qui passe de 5 à 16 km/h. La fenêtre exploitable se situe avant 10 h, toute l’année.

In [158]:
# Croisement mois × heure : part d'heures avec vent offshore
pd.crosstab(
    df["mois"],
    df["heure"],
    values=df["offshore"],
    aggfunc="mean",
).round(2) * 100


heure,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
mois,,,,,,,,,,,,,,,,,,,,,
1,63.0,71.0,68.0,71.0,66.0,65.0,60.0,66.0,69.0,69.0,...,2.0,0.0,0.0,0.0,2.0,6.0,16.0,24.0,44.0,48.0
2,47.0,46.0,37.0,51.0,56.0,58.0,58.0,58.0,58.0,60.0,...,2.0,2.0,0.0,2.0,2.0,4.0,12.0,16.0,21.0,26.0
3,32.0,35.0,35.0,44.0,32.0,29.0,29.0,37.0,44.0,45.0,...,0.0,0.0,0.0,0.0,0.0,2.0,3.0,13.0,18.0,23.0
4,27.0,37.0,27.0,27.0,27.0,25.0,20.0,15.0,22.0,8.0,...,0.0,2.0,0.0,0.0,0.0,3.0,3.0,5.0,13.0,18.0
5,8.0,8.0,10.0,15.0,10.0,13.0,16.0,21.0,19.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,5.0
6,10.0,8.0,12.0,12.0,13.0,12.0,17.0,20.0,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,5.0
7,5.0,3.0,6.0,16.0,23.0,19.0,21.0,27.0,16.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,6.0
8,5.0,11.0,18.0,18.0,26.0,32.0,23.0,39.0,32.0,3.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,5.0,8.0,5.0
9,22.0,17.0,35.0,27.0,25.0,30.0,35.0,35.0,35.0,12.0,...,0.0,0.0,0.0,0.0,2.0,2.0,2.0,12.0,3.0,8.0


Les meilleures fenêtres sont les matinées de novembre à février (60-73 % offshore). L’après-midi est inexploitable quel que soit le mois.

In [159]:
# Répartition des houles par secteur de provenance
df["secteur_houle"] = pd.cut(
    df["wave_direction"],
    bins=[0, 45, 90, 135, 180, 225, 270, 315, 360],
    labels=["N-NE", "NE-E", "E-SE", "SE-S", "S-SO", "SO-O", "O-NO", "NO-N"],
)

df["secteur_houle"].value_counts(normalize=True).round(3) * 100

secteur_houle
NO-N    65.7
O-NO    33.9
SO-O     0.2
E-SE     0.1
SE-S     0.1
S-SO     0.0
N-NE     0.0
NE-E     0.0
Name: proportion, dtype: float64

99,6 % des houles viennent du secteur nord-ouest à ouest. Cette variable ne discrimine donc rien et sera écartée du score de qualité.

In [160]:
# Répartition des hauteurs de vague par tranche
df["tranche_vague"] = pd.cut(
    df["wave_height"],
    bins=[0, 0.8, 1.2, 1.8, 2.5, 10],
    labels=["plat (<0.8m)", "petit (0.8-1.2)", "correct (1.2-1.8)", "gros (1.8-2.5)", "très gros (>2.5)"],
)

df["tranche_vague"].value_counts(normalize=True).round(3) * 100

tranche_vague
petit (0.8-1.2)      40.4
correct (1.2-1.8)    27.5
plat (<0.8m)         21.8
gros (1.8-2.5)        8.6
très gros (>2.5)      1.6
Name: proportion, dtype: float64

36 % du temps en taille exploitable, 62 % en petit ou plat, 1,6 % de très gros. Taghazout est un spot accessible, cohérent avec sa réputation d’école de surf.

In [161]:
# Écart entre la vague totale et la houle longue : le reste est du clapot local
df["part_houle"] = (df["swell_wave_height"] / df["wave_height"]).round(2)

df["part_houle"].describe()

count    17544.000000
mean         0.851827
std          0.137622
min          0.100000
25%          0.780000
50%          0.890000
75%          0.960000
max          1.000000
Name: part_houle, dtype: float64

In [162]:
# Une heure est surfable si les trois critères sont réunis
df["surfable"] = (
    (df["wave_height"] >= 1.0)      # taille suffisante
    & (df["part_houle"] >= 0.80)    # mer propre
    & (df["offshore"])              # vent favorable
)

# Part d'heures surfables par mois, en %
(df.groupby("mois")["surfable"].mean() * 100).round(1)

mois
1     25.7
2     14.2
3     12.3
4      7.2
5      2.1
6      0.8
7      1.3
8      2.9
9      4.7
10     7.5
11    13.4
12    15.3
Name: surfable, dtype: float64

En croisant les trois critères (taille ≥ 1 m, part de houle ≥ 0,80, vent offshore), seules 9 % des heures de l’année sont réellement surfables. Janvier concentre le quart de ses heures en conditions favorables, contre moins de 1 % en juin. Les critères pris isolément étaient trompeurs : c’est leur conjonction qui est rare.

In [163]:
part = (df["surfable"].mean() * 100).round(1)
print(f"Sur les deux années, {part} % des heures réunissent les trois critères de surf.")

Sur les deux années, 9.0 % des heures réunissent les trois critères de surf.


In [164]:
# Part d'heures surfables, année par année
df["annee"] = pd.to_datetime(df["time"]).dt.year
(df.groupby("annee")["surfable"].mean() * 100).round(1)

annee
2024    9.1
2025    8.8
Name: surfable, dtype: float64

Les deux années donnent 9,1 % et 8,8 % d’heures surfables. La régularité inter-annuelle confirme que les moyennes mensuelles sont représentatives et non tirées par une saison exceptionnelle.

In [165]:
(df.groupby("heure")["surfable"].mean() * 100).round(1)

heure
0     14.9
1     15.2
2     14.9
3     17.8
4     17.5
5     18.9
6     17.1
7     18.3
8     18.6
9     15.7
10     7.3
11     1.9
12     1.2
13     0.7
14     0.4
15     0.5
16     0.0
17     0.1
18     0.3
19     1.9
20     3.1
21     7.5
22     9.6
23    11.4
Name: surfable, dtype: float64

Répartition par heure sur les 24 h. Le pic se situe entre 3 h et 9 h (17 à 19 % d’heures surfables), puis l’effondrement est brutal : 7,3 % à 10 h, moins de 1 % l’après-midi.

Problème : cette lecture inclut les heures nocturnes (0 h-5 h), où les conditions peuvent être bonnes mais la pratique impossible faute de lumière. Ces heures gonflent artificiellement le résultat. Il faut restreindre le calcul aux heures de jour avant de conclure.

In [166]:
# On ne garde que les heures de jour (6h-19h)
jour = df[df["heure"].between(6, 19)]

print(f"Part d'heures surfables en journée : {(jour['surfable'].mean() * 100).round(1)} %")

Part d'heures surfables en journée : 6.0 %


En se limitant de 6h à 19h, la fenêtre réellement exploitable est de 6h à 9h. Dès 10 h la brise thermique se lève et dégrade les conditions pour le reste de la journée.

In [167]:
# Écart entre vent moyen et rafales, par heure de la journée
df.groupby("heure")[["wind_speed_10m", "wind_gusts_10m"]].mean().round(1)

,wind_speed_10m,wind_gusts_10m
heure,,
0,6.8,17.9
1,6.1,16.3
2,5.9,16.1
3,5.6,14.9
4,5.2,14.2
5,5.1,13.6
6,5.2,13.5
7,5.2,13.4
8,5.4,14.1


Les rafales représentent systématiquement environ 2,4 fois le vent moyen, quel que soit le moment de la journée. Cette proportion constante signifie que la variable n’ajoute rien à l’analyse : elle est redondante avec le vent moyen et sera écartée du score.

In [168]:
colonnes = ["wave_height", "wave_period", "swell_wave_period",
            "wind_speed_10m", "temperature_2m", "part_houle"]

df[colonnes].corr().round(2)

,wave_height,wave_period,swell_wave_period,wind_speed_10m,temperature_2m,part_houle
wave_height,1.00,0.44,0.45,0.24,-0.30,0.07
wave_period,0.44,1.00,0.89,-0.15,-0.31,0.29
swell_wave_period,0.45,0.89,1.00,0.08,-0.27,0.02
wind_speed_10m,0.24,-0.15,0.08,1.00,0.23,-0.50
temperature_2m,-0.30,-0.31,-0.27,0.23,1.00,-0.07
part_houle,0.07,0.29,0.02,-0.50,-0.07,1.00


Trois enseignements. D’abord, wave_period et swell_wave_period sont corrélées à 0,89 : une seule suffit. Ensuite, vent et propreté de la mer sont liés à −0,50, ce qui confirme que le vent local fabrique le clapot — il ne faut donc pas compter deux fois cet effet dans le score. Enfin, la corrélation négative entre température et hauteur de vague (−0,30) chiffre l’opposition entre saison de surf et saison balnéaire.

In [169]:
# Les heures de très grosse houle, par mois
gros = df[df["wave_height"] > 2.5]
gros.groupby("mois").size()

mois
1      23
2      67
3     171
12     28
dtype: int64

In [170]:
# Grosses houles pendant les heures de jour (6h-19h)
gros_jour = df[(df["wave_height"] > 2.5) & (df["heure"].between(6, 19))]
print("En journée :")
print(gros_jour.groupby("mois").size())

# Grosses houles pendant la fenêtre exploitable (6h-9h)
gros_fenetre = df[(df["wave_height"] > 2.5) & (df["heure"].between(6, 9))]
print("\nFenêtre 6h-9h :")
print(gros_fenetre.groupby("mois").size())

En journée :
mois
1      11
2      33
3     100
12     17
dtype: int64

Fenêtre 6h-9h :
mois
1      4
2      8
3     26
12     7
dtype: int64


In [171]:
# Une heure est agréable pour un touriste : chaud, sec, peu de vent, mer calme
jour = df[df["heure"].between(9, 18)]

jour = jour.assign(
    agreable=(
        jour["temperature_2m"].between(20, 32)
        & (jour["precipitation"] == 0)
        & (jour["wind_speed_10m"] < 20)
        & (jour["wave_height"] < 1.2)
    )
)

(jour.groupby("mois")["agreable"].mean() * 100).round(1)

mois
1     17.7
2     21.6
3     14.7
4     32.0
5     56.6
6     55.5
7     61.1
8     47.9
9     58.3
10    61.1
11    45.7
12    29.4
Name: agreable, dtype: float64

In [172]:
# Pourquoi août décroche : on regarde chaque critère séparément
aout = jour[jour["mois"] == 8]

print(f"Trop chaud (>32°C) : {(aout['temperature_2m'] > 32).mean() * 100:.1f} %")
print(f"Trop venté (>20 km/h) : {(aout['wind_speed_10m'] > 20).mean() * 100:.1f} %")
print(f"Mer agitée (>1.2m) : {(aout['wave_height'] > 1.2).mean() * 100:.1f} %")

Trop chaud (>32°C) : 31.8 %
Trop venté (>20 km/h) : 8.7 %
Mer agitée (>1.2m) : 16.1 %


Le confort touristique suit le cycle inverse du surf : 48 à 61 % d’heures agréables de mai à octobre, contre 15 à 22 % de janvier à mars. Octobre égale juillet (61,1 %), ce qui fait de l’arrière-saison une période aussi favorable que le plein été. Le creux d’août (47,9 %) s’explique par la chaleur : 31,8 % des heures y dépassent 32 °C, contre seulement 8,7 % de vent fort. Ce seuil de 32 °C reste un choix discutable, un visiteur venu pour la plage pouvant juger ces conditions acceptables.

In [173]:
df.columns.tolist()

['time',
 'wave_height',
 'wave_period',
 'wave_direction',
 'swell_wave_height',
 'swell_wave_period',
 'sea_surface_temperature',
 'temperature_2m',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'precipitation',
 'relative_humidity_2m',
 'dew_point_2m',
 'cloud_cover_low',
 'mois',
 'offshore',
 'heure',
 'secteur_houle',
 'tranche_vague',
 'part_houle',
 'surfable',
 'annee']

In [174]:
df = collecter_historique()
df[["time", "sea_surface_temperature"]].describe()

,sea_surface_temperature
count,17232.000000
mean,19.487134
std,1.529737
min,15.700000
25%,18.200000
50%,19.500000
75%,20.800000
max,23.900000


In [175]:
# Température de l'eau par mois
df["mois"] = pd.to_datetime(df["time"]).dt.month
df.groupby("mois")["sea_surface_temperature"].mean().round(1)

mois
1     18.1
2     17.8
3     17.5
4     18.7
5     19.0
6     19.7
7     20.6
8     21.9
9     21.0
10    20.9
11    19.9
12    18.3
Name: sea_surface_temperature, dtype: float64

La température de l’eau varie peu : 17,5 °C en mars à 21,9 °C en août, soit 4,4 °C d’amplitude annuelle. Le pic survient en août-septembre, avec un mois de décalage sur la température de l’air — l’inertie thermique de l’océan. L’eau ne dépassant jamais 22 °C en moyenne, la baignade reste fraîche même en plein été et la combinaison est nécessaire toute l’année pour le surf.

Donnée incomplète : 312 valeurs manquantes sur 17 544 (1,8 %), seule variable du jeu à en comporter.

In [176]:
df[df["sea_surface_temperature"].isna()]["time"].min(), df[df["sea_surface_temperature"].isna()]["time"].max()

('2025-01-29T07:00', '2025-02-11T06:00')

Les 312 valeurs manquantes forment un bloc continu du 29 janvier au 11 février 2025. Il ne s’agit donc pas de valeurs aléatoirement absentes mais d’une interruption de la source sur treize jours. Cette période sera exclue des moyennes de température de l’eau, les autres variables restant exploitables.

In [177]:
# Recrée les colonnes dérivées après rechargement
df["mois"] = pd.to_datetime(df["time"]).dt.month
df["heure"] = pd.to_datetime(df["time"]).dt.hour
df["annee"] = pd.to_datetime(df["time"]).dt.year
df["offshore"] = df["wind_direction_10m"].between(45, 135)
df["part_houle"] = (df["swell_wave_height"] / df["wave_height"]).round(2)

df["surfable"] = (
    (df["wave_height"] >= 1.0)
    & (df["part_houle"] >= 0.80)
    & (df["offshore"])
)

In [178]:
# Confort touristique en tenant compte de la température de l'eau
jour = df[df["heure"].between(6, 19)]

jour = jour.assign(
    baignade=(
        jour["temperature_2m"].between(20, 32)
        & (jour["precipitation"] == 0)
        & (jour["wind_speed_10m"] < 20)
        & (jour["wave_height"] < 1.2)
        & (jour["sea_surface_temperature"] >= 20)
    )
)

(jour.groupby("mois")["baignade"].mean() * 100).round(1)

mois
1      0.0
2      0.0
3      0.0
4      1.4
5      0.5
6     19.0
7     46.5
8     49.8
9     53.3
10    45.0
11     8.8
12     3.3
Name: baignade, dtype: float64

En intégrant une température d’eau d’au moins 20 °C, la saison balnéaire se resserre à juin-octobre. Septembre devient le meilleur mois (53,3 %), devant août (49,8 %) et juillet (46,5 %) : l’eau atteint son maximum thermique avec un mois de décalage sur l’air, tandis que les excès de chaleur de l’été s’estompent. De novembre à mai, la baignade confortable est pratiquement impossible malgré un air parfois agréable.

In [179]:
# Brouillard : air saturé et écart minime avec le point de rosée
df["brouillard"] = (
    (df["relative_humidity_2m"] >= 95)
    & ((df["temperature_2m"] - df["dew_point_2m"]) <= 1.0)
)

# Part d'heures de brouillard par mois, en %
(df.groupby("mois")["brouillard"].mean() * 100).round(1)

mois
1      0.5
2      2.0
3      5.6
4      5.9
5     10.1
6     10.0
7     10.3
8      9.5
9      8.8
10    14.5
11     1.7
12     1.7
Name: brouillard, dtype: float64

In [180]:
(df.groupby("heure")["brouillard"].mean() * 100).round(1)

heure
0      5.3
1     10.9
2     15.7
3     18.9
4     21.9
5     21.5
6     20.4
7     19.4
8     10.9
9      5.5
10     1.2
11     0.3
12     0.3
13     0.1
14     0.1
15     0.1
16     0.1
17     0.0
18     0.0
19     0.5
20     1.4
21     1.6
22     2.5
23     3.3
Name: brouillard, dtype: float64

In [181]:
# Parmi les heures surfables, quelle part est gâchée par le brouillard ?
surfables = df[df["surfable"] & df["heure"].between(6, 19)]

print(f"Heures surfables en journée : {len(surfables)}")
print(f"Dont avec brouillard : {(surfables['brouillard'].mean() * 100).round(1)} %")

Heures surfables en journée : 615
Dont avec brouillard : 2.9 %


Le brouillard culmine entre 4h et 7h (jusqu’à 22 % des heures), soit la fenêtre même où les conditions de surf sont les meilleures. On pourrait en conclure à un conflit. Le croisement montre l’inverse : seules 2,9 % des 615 heures surfables de journée sont brumeuses. La raison est mécanique — le brouillard est apporté par l’air marin humide, donc par vent de mer, tandis que le critère de surf exige un vent de terre sec. Le vent offshore qui lisse la houle chasse simultanément la brume.

Pour le tourisme en revanche, l’effet est réel : octobre, mois le plus favorable à la baignade, est aussi le plus brumeux (14,5 % des heures), avec des matinées fréquemment bouchées avant dissipation vers 10h.

In [182]:
# Ciel bouché : plus de 70 % de nuages bas, en journée
jour = df[df["heure"].between(9, 18)]
couvert = (jour["cloud_cover_low"] > 70)

(jour.assign(couvert=couvert).groupby("mois")["couvert"].mean() * 100).round(1)

mois
1      0.6
2      3.2
3     11.5
4      5.3
5      7.1
6      7.7
7      5.6
8      3.2
9      5.8
10     6.6
11     2.3
12     4.5
Name: couvert, dtype: float64

La nébulosité basse dépasse 70 % dans moins de 8 % des heures de journée, sauf en mars (11,5 %). Le ciel couvert n’est donc pas un facteur discriminant à Taghazout : la variable est écartée de l’analyse.

In [183]:
# Jours de pluie par mois et cumul moyen
pluie = df.groupby("mois")["precipitation"].agg(
    heures_de_pluie=lambda x: (x > 0).sum(),
    cumul_mm="sum",
).round(1)

pluie

,heures_de_pluie,cumul_mm
mois,,
1,7,1.2
2,46,66.8
3,334,151.6
4,66,35.6
5,12,1.9
6,5,0.5
7,0,0.0
8,0,0.0
9,19,4.2


Les précipitations se concentrent sur mars (334 heures de pluie et 151,6 mm cumulés sur deux ans, soit un tiers du total annuel), mois également marqué par les grosses houles : ce sont les mêmes dépressions atlantiques. Juillet et août n’enregistrent aucune précipitation sur toute la période. Les cumuls portant sur deux années, la moyenne annuelle de mars avoisine 76 mm, ce qui reste faible et confirme le caractère semi-aride du climat.

In [184]:
from collecte import collecter_previsions

prev = collecter_previsions()
print(prev.shape)
print(prev.columns.tolist())
prev.describe().round(1)

(168, 15)
['time', 'wave_height', 'wave_period', 'wave_direction', 'swell_wave_height', 'swell_wave_period', 'sea_surface_temperature', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'precipitation', 'relative_humidity_2m', 'dew_point_2m', 'cloud_cover_low']


,wave_height,wave_period,wave_direction,swell_wave_height,swell_wave_period,sea_surface_temperature,temperature_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation,relative_humidity_2m,dew_point_2m,cloud_cover_low
count,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0,168.0
mean,0.9,10.0,321.5,0.8,8.9,20.9,27.3,7.9,130.0,16.2,0.0,40.6,12.5,0.0
std,0.2,0.8,6.2,0.2,1.0,0.2,2.6,2.9,107.9,5.4,0.0,6.9,2.2,0.0
min,0.6,9.0,312.0,0.4,7.0,20.6,22.4,0.8,4.0,6.5,0.0,25.0,6.5,0.0
25%,0.7,9.2,316.0,0.7,8.1,20.8,25.2,6.2,45.0,12.2,0.0,35.8,11.5,0.0
50%,0.9,9.9,321.0,0.8,8.8,20.9,27.2,8.0,73.5,15.3,0.0,40.0,12.6,0.0
75%,1.1,10.7,328.0,1.0,9.4,21.0,29.1,9.8,237.5,18.5,0.0,46.0,13.4,0.0
max,1.4,11.6,330.0,1.3,10.6,21.3,33.0,18.4,360.0,32.8,0.1,58.0,18.3,0.0


In [185]:
prev["heure"] = pd.to_datetime(prev["time"]).dt.hour
prev["offshore"] = prev["wind_direction_10m"].between(45, 135)
prev["part_houle"] = (prev["swell_wave_height"] / prev["wave_height"]).round(2)

prev["surfable"] = (
    (prev["wave_height"] >= 1.0)
    & (prev["part_houle"] >= 0.80)
    & prev["offshore"]
)

print(f"Heures surfables prévues : {prev['surfable'].sum()} sur 168")

Heures surfables prévues : 13 sur 168


In [186]:
prev[prev["surfable"]][["time", "wave_height", "wind_speed_10m"]]

,time,wave_height,wind_speed_10m
0,2026-10-25T00:00,1.14,6.1
81,2026-10-28T09:00,1.00,3.3
82,2026-10-28T10:00,1.02,1.1
98,2026-10-29T02:00,1.38,7.4
99,2026-10-29T03:00,1.36,7.9
100,2026-10-29T04:00,1.36,8.4
101,2026-10-29T05:00,1.34,9.9
102,2026-10-29T06:00,1.34,11.2
115,2026-10-29T19:00,1.06,3.3
116,2026-10-29T20:00,1.04,4.7


La table des prévisions partage la structure de l’historique et se prête aux mêmes règles. Sur les 168 heures à venir, 13 remplissent les critères (7,7 %, proche de la moyenne annuelle de 9 %), mais seules 3 tombent dans la plage 6h-19h. Le filtre horaire doit donc être appliqué dès la couche gold pour ne pas surestimer les créneaux annoncés.